# 03 · Temporal State：从 observation 到稳定的 agent state

本章合并旧版 `00D` 和 `07`。单帧 perception 是 observation，不是 world state。prediction 需要的是带有 `track_id、位置、速度、年龄、不确定性` 的时序状态，还需要考虑 ego-motion、dropout、outlier 和 timestamp offset。

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ad_tutorial import (
    ARTIFACT_DIR,
    BEVConfig,
    build_bev_dataset,
    build_urban_cut_in_scene,
    ensure_artifact_dir,
    load_json_artifact,
    load_numpy_artifact,
    save_json_artifact,
    save_numpy_artifact,
    scene_to_bev,
)

ensure_artifact_dir()
print("project root:", PROJECT_ROOT)
print("artifact directory:", ARTIFACT_DIR)

import numpy as np
import matplotlib.pyplot as plt

times = np.arange(0.0, 8.0, 0.1)
truth = np.array([build_urban_cut_in_scene(seed=7, timestamp_s=float(t)).cut_in_xy for t in times])
rng = np.random.default_rng(12)
observation = truth + rng.normal(0.0, 0.35, truth.shape)
dropout = (times >= 3.0) & (times < 4.2)
observation[dropout] = np.nan
observation[58] += np.array([2.2, -1.5])

def smooth_track(values, alpha=0.25):
    estimate = []
    current = values[0].copy()
    for value in values:
        if np.isfinite(value).all():
            current = alpha * value + (1.0 - alpha) * current
        estimate.append(current.copy())
    return np.asarray(estimate)

estimate = smooth_track(observation)
velocity = np.gradient(estimate, times, axis=0)
uncertainty = np.where(np.isfinite(observation).all(axis=1), 0.35, 0.9)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(truth[:, 0], truth[:, 1], label="latent cut-in")
axes[0].scatter(observation[:, 0], observation[:, 1], s=8, alpha=0.35, label="observation")
axes[0].plot(estimate[:, 0], estimate[:, 1], label="tracked state")
axes[0].set_aspect("equal")
axes[0].legend()
axes[0].set(title="Association/state estimation in the ego frame", xlabel="x / m", ylabel="y / m")
axes[1].plot(times, velocity[:, 1], label="estimated lateral velocity")
axes[1].fill_between(times, -uncertainty, uncertainty, alpha=0.2, label="uncertainty proxy")
axes[1].axvspan(3.0, 4.2, color="orange", alpha=0.15, label="observation dropout")
axes[1].legend()
axes[1].set(title="State memory changes prediction input", xlabel="time / s", ylabel="m/s")
plt.tight_layout()
plt.show()

valid = ~dropout
rmse = float(np.sqrt(np.mean((estimate[valid] - truth[valid]) ** 2)))
print({"track_rmse_m": round(rmse, 3), "max_dropout_frames": int(dropout.sum()), "last_velocity": velocity[-1].round(3).tolist()})

## Tracking is an interface, not a magic filter

A Kalman filter, learned tracker, or transformer memory can implement the update. The system still needs to decide what to do when association is ambiguous: keep a stale track, reduce confidence, create a new ID, or trigger a safety/degraded mode. Tracking mistakes can be amplified by prediction and planning.

In [ ]:
from ipywidgets import FloatSlider, interact

def tracking_ablation(alpha=0.25, dropout_duration=1.0, outlier_m=2.0):
    local_truth = np.array([build_urban_cut_in_scene(seed=7, timestamp_s=float(t)).cut_in_xy for t in times])
    local_obs = local_truth + np.random.default_rng(18).normal(0, 0.35, local_truth.shape)
    local_dropout = (times >= 3.0) & (times < 3.0 + dropout_duration)
    local_obs[local_dropout] = np.nan
    local_obs[np.argmin(abs(times - 5.8))] += np.array([outlier_m, -0.5 * outlier_m])
    local_est = smooth_track(local_obs, alpha=alpha)
    score = np.sqrt(np.mean((local_est[~local_dropout] - local_truth[~local_dropout]) ** 2))
    print(f"alpha={alpha:.2f}, dropout={dropout_duration:.1f}s, outlier={outlier_m:.1f}m -> RMSE={score:.3f}m")

interact(
    tracking_ablation,
    alpha=FloatSlider(min=0.05, max=0.8, step=0.05, value=0.25),
    dropout_duration=FloatSlider(min=0.0, max=2.5, step=0.1, value=1.0),
    outlier_m=FloatSlider(min=0.0, max=5.0, step=0.25, value=2.0),
)

In [ ]:
save_numpy_artifact(
    "03_temporal_state.npz",
    time_s=times,
    truth_xy=truth,
    observation_xy=np.nan_to_num(observation, nan=-999.0),
    estimate_xy=estimate,
    velocity_xy=velocity,
    uncertainty=uncertainty,
)
save_json_artifact("03_temporal_state_meta.json", {
    "track_id": "cut_in_0",
    "fields": ["position_xy", "velocity_xy", "age", "uncertainty"],
    "dropout_frames": int(dropout.sum()),
    "next": "04_localization_mapping.ipynb",
})
print("saved temporal state artifact")

### 完成标准

解释 ego-motion compensation 与 actor velocity estimation 的区别；说明 dropout 期间你会怎样设置 track age/uncertainty；并说出 tracking error 为什么会改变 prediction 的 miss rate。下一章单独处理 ego pose、漂移和地图匹配，避免把 actor state 与 localization 混为一谈。